In [1]:
import pandas as pd

df = pd.read_csv("ai4i2020.csv")
print("حجم الداتا:", df.shape)
print("\nالأعمدة:")
print(df.columns.tolist())
print("\n--- أول 5 صفوف ---")
print(df.head())

حجم الداتا: (10000, 14)

الأعمدة:
['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

--- أول 5 صفوف ---
   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    1     M14860    M                298.1                    308.6   
1    2     L47181    L                298.2                    308.7   
2    3     L47182    L                298.1                    308.5   
3    4     L47183    L                298.2                    308.6   
4    5     L47184    L                298.2                    308.7   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
0                    1551         42.8                0                0    0   
1                    1408         46.3                3                0    0   
2                    1498         49.4                5                0    

In [2]:

# ----------------------------------------------------
# نخزن أعمدة الـ Target جنبًا (للتحقق لاحقًا، مو للتدريب) قبل ما نحذفها من الداتا الشغالة
ground_truth = df[['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']].copy()

# ----------------------------------------------------
# نبني الداتا الشغالة: نحذف المعرّفات + كل أعمدة الـ Target دفعة وحدة
work_df = df.drop(['UDI', 'Product ID', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'], axis=1)

print("أعمدة work_df:", work_df.columns.tolist())
print("شكلها:", work_df.shape)

# ----------------------------------------------------
# Ordinal Encoding لعمود Type - نفس منطق مشروع الألماس بالضبط
quality_order = ['L', 'M', 'H']   # من الأقل جودة للأعلى
work_df['Type'] = pd.Categorical(work_df['Type'], categories=quality_order, ordered=True)
work_df['Type'] = work_df['Type'].cat.codes

print("\n--- أول 5 صفوف بعد المعالجة ---")
print(work_df.head())

print("\n--- توزيع Type بعد التحويل ---")
print(work_df['Type'].value_counts().sort_index())

أعمدة work_df: ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
شكلها: (10000, 6)

--- أول 5 صفوف بعد المعالجة ---
   Type  Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
0     1                298.1                    308.6                    1551   
1     0                298.2                    308.7                    1408   
2     0                298.1                    308.5                    1498   
3     0                298.2                    308.6                    1433   
4     0                298.2                    308.7                    1408   

   Torque [Nm]  Tool wear [min]  
0         42.8                0  
1         46.3                3  
2         49.4                5  
3         39.5                7  
4         40.0                9  

--- توزيع Type بعد التحويل ---
Type
0    6000
1    2997
2    1003
Name: count, dtype: int64


In [5]:
from sklearn.preprocessing import StandardScaler

# ----------------------------------------------------
# نطبق Scaling على كل أعمدة work_df (تشمل Type بعد الترميز، والأعمدة الرقمية الخمسة)
scaler = StandardScaler()
work_df_scaled = scaler.fit_transform(work_df)

# نحولها رجوع لـ DataFrame بنفس أسماء الأعمدة، عشان تضل مقروءة وواضحة
work_df_scaled = pd.DataFrame(work_df_scaled, columns=work_df.columns, index=work_df.index)

print("--- قبل Scaling ---")
print(work_df.describe().round(2))

print("\n--- بعد Scaling ---")
print(work_df_scaled.describe().round(2))

--- قبل Scaling ---
           Type  Air temperature [K]  Process temperature [K]  \
count  10000.00              10000.0                 10000.00   
mean       0.50                300.0                   310.01   
std        0.67                  2.0                     1.48   
min        0.00                295.3                   305.70   
25%        0.00                298.3                   308.80   
50%        0.00                300.1                   310.10   
75%        1.00                301.5                   311.10   
max        2.00                304.5                   313.80   

       Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  
count                10000.00     10000.00         10000.00  
mean                  1538.78        39.99           107.95  
std                    179.28         9.97            63.65  
min                   1168.00         3.80             0.00  
25%                   1423.00        33.20            53.00  
50%                   

In [6]:
# ----------------------------------------------------
# نحدد الصفوف "المتطرفة" بعمود Rotational speed بعد Scaling
# مثلاً: أي صف قيمته المُوحّدة أكبر من 3 (يعني أبعد من 3 انحرافات معيارية عن المتوسط)
extreme_speed_mask = work_df_scaled['Rotational speed [rpm]'] > 3

print(f"عدد الصفوف المتطرفة بـ Rotational speed: {extreme_speed_mask.sum()}")
print(f"نسبتهم من كل الداتا: {extreme_speed_mask.sum() / len(work_df) * 100:.2f}%")

# ----------------------------------------------------
# نستخدم ground_truth (المحفوظ جنبًا) للتحقق - بدون ما يكون أثر على أي قرار سابق
print("\n--- نسبة Machine Failure بين الصفوف المتطرفة ---")
print(ground_truth[extreme_speed_mask]['Machine failure'].value_counts(normalize=True).round(3))

print("\n--- نسبة Machine Failure بين كل الصفوف (كمرجع للمقارنة) ---")
print(ground_truth['Machine failure'].value_counts(normalize=True).round(3))

عدد الصفوف المتطرفة بـ Rotational speed: 164
نسبتهم من كل الداتا: 1.64%

--- نسبة Machine Failure بين الصفوف المتطرفة ---
Machine failure
0    0.799
1    0.201
Name: proportion, dtype: float64

--- نسبة Machine Failure بين كل الصفوف (كمرجع للمقارنة) ---
Machine failure
0    0.966
1    0.034
Name: proportion, dtype: float64


In [9]:
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import LocalOutlierFactor
import numpy as np

results_comparison = pd.DataFrame(index=work_df.index)

# ----------------------------------------------------
# 1) Isolation Forest
# contamination=0.034 -> نعطيها تقريبًا نفس نسبة الشذوذ الحقيقية المعروفة من التوثيق (3.4%)
# هيك منقارنها بعدالة مع باقي الطرق، بدل ما تخمن نسبة عشوائية
iso_forest = IsolationForest(contamination=0.034, random_state=42, n_jobs=-1)
results_comparison['IsolationForest'] = iso_forest.fit_predict(work_df_scaled)
# fit_predict بترجع: -1 = شاذ، 1 = طبيعي

# ----------------------------------------------------
# 2) DBSCAN
# eps و min_samples قيم معقولة كبداية لبيانات 5-6 أبعاد بعد Scaling
dbscan = DBSCAN(eps=1.2, min_samples=10)
results_comparison['DBSCAN'] = dbscan.fit_predict(work_df_scaled)
# fit_predict بترجع: -1 = شاذ (Noise)، أي رقم ≥0 = رقم المجموعة

# ----------------------------------------------------
# 3) K-Means + مسافة عن المركز
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(work_df_scaled)
# نحسب المسافة الإقليدية من كل نقطة لمركز مجموعتها
distances = np.min(kmeans.transform(work_df_scaled), axis=1)
# نعتبر أبعد 3.4% من النقاط (نفس النسبة المرجعية) كشاذة
threshold = np.percentile(distances, 100 - 3.4)
results_comparison['KMeans'] = np.where(distances > threshold, -1, 1)

# ----------------------------------------------------
# 4) Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.034, n_jobs=-1)
results_comparison['LOF'] = lof.fit_predict(work_df_scaled)
# fit_predict بترجع: -1 = شاذ، 1 = طبيعي

print("عدد الحالات المصنّفة 'شاذة' من كل خوارزمية:")
for col in results_comparison.columns:
    n_anomaly = (results_comparison[col] == -1).sum()
    print(f"  {col}: {n_anomaly} حالة ({n_anomaly/len(work_df)*100:.2f}%)")

عدد الحالات المصنّفة 'شاذة' من كل خوارزمية:
  IsolationForest: 340 حالة (3.40%)
  DBSCAN: 88 حالة (0.88%)
  KMeans: 340 حالة (3.40%)
  LOF: 340 حالة (3.40%)


In [10]:
# ----------------------------------------------------
# نضيف عمود Machine failure الحقيقي (من ground_truth) للمقارنة فقط - بعد ما خلصنا كل التحليل
results_comparison['Actual_Failure'] = ground_truth['Machine failure'].values

print("\n--- تقاطع كل خوارزمية مع الواقع الفعلي (Machine failure) ---")
for col in ['IsolationForest', 'DBSCAN', 'KMeans', 'LOF']:
    detected_anomaly = results_comparison[col] == -1
    actual_failure_rate = results_comparison[detected_anomaly]['Actual_Failure'].mean()
    print(f"{col}: من الحالات المكتشفة كشاذة، {actual_failure_rate*100:.1f}% فعليًا عطل حقيقي")


--- تقاطع كل خوارزمية مع الواقع الفعلي (Machine failure) ---
IsolationForest: من الحالات المكتشفة كشاذة، 17.1% فعليًا عطل حقيقي
DBSCAN: من الحالات المكتشفة كشاذة، 42.0% فعليًا عطل حقيقي
KMeans: من الحالات المكتشفة كشاذة، 23.5% فعليًا عطل حقيقي
LOF: من الحالات المكتشفة كشاذة، 24.1% فعليًا عطل حقيقي


In [11]:
from sklearn.metrics import f1_score

# ----------------------------------------------------
# دالة تقييم موحّدة: تحول أي نتيجة خوارزمية لـ"شاذ=1 / طبيعي=0" وتقارنها بالواقع
def evaluate_anomaly_detection(predicted_labels, actual_failure):
    predicted_binary = (predicted_labels == -1).astype(int)
    return f1_score(actual_failure, predicted_binary)

actual = ground_truth['Machine failure'].values

# ==========================================
# 1) Isolation Forest - نجرب عدة قيم contamination
# ==========================================
best_iso_score, best_iso_params = 0, None
for contamination in [0.01, 0.02, 0.034, 0.05, 0.1]:
    model = IsolationForest(contamination=contamination, random_state=42, n_jobs=-1)
    pred = model.fit_predict(work_df_scaled)
    score = evaluate_anomaly_detection(pred, actual)
    if score > best_iso_score:
        best_iso_score, best_iso_params = score, {'contamination': contamination}

print(f"Isolation Forest أفضل: {best_iso_params}, F1={best_iso_score:.3f}")

# ==========================================
# 2) DBSCAN - نجرب تركيبات eps و min_samples
# ==========================================
best_dbscan_score, best_dbscan_params = 0, None
for eps in [0.8, 1.0, 1.2, 1.5, 2.0]:
    for min_samples in [5, 10, 15, 20]:
        model = DBSCAN(eps=eps, min_samples=min_samples)
        pred = model.fit_predict(work_df_scaled)
        score = evaluate_anomaly_detection(pred, actual)
        if score > best_dbscan_score:
            best_dbscan_score, best_dbscan_params = score, {'eps': eps, 'min_samples': min_samples}

print(f"DBSCAN أفضل: {best_dbscan_params}, F1={best_dbscan_score:.3f}")

# ==========================================
# 3) K-Means + مسافة عن المركز - نجرب K ونسبة القطع (percentile)
# ==========================================
best_kmeans_score, best_kmeans_params = 0, None
for k in [2, 3, 4, 5, 6]:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(work_df_scaled)
    distances = np.min(model.transform(work_df_scaled), axis=1)
    for pct in [90, 93, 95, 97, 99]:
        threshold = np.percentile(distances, pct)
        pred = np.where(distances > threshold, -1, 1)
        score = evaluate_anomaly_detection(pred, actual)
        if score > best_kmeans_score:
            best_kmeans_score, best_kmeans_params = score, {'k': k, 'percentile': pct}

print(f"K-Means أفضل: {best_kmeans_params}, F1={best_kmeans_score:.3f}")

# ==========================================
# 4) LOF - نجرب n_neighbors وcontamination
# ==========================================
best_lof_score, best_lof_params = 0, None
for n_neighbors in [10, 20, 30, 50]:
    for contamination in [0.02, 0.034, 0.05, 0.1]:
        model = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination, n_jobs=-1)
        pred = model.fit_predict(work_df_scaled)
        score = evaluate_anomaly_detection(pred, actual)
        if score > best_lof_score:
            best_lof_score, best_lof_params = score, {'n_neighbors': n_neighbors, 'contamination': contamination}

print(f"LOF أفضل: {best_lof_params}, F1={best_lof_score:.3f}")

Isolation Forest أفضل: {'contamination': 0.05}, F1=0.188
DBSCAN أفضل: {'eps': 0.8, 'min_samples': 5}, F1=0.218
K-Means أفضل: {'k': 5, 'percentile': 95}, F1=0.241
LOF أفضل: {'n_neighbors': 50, 'contamination': 0.05}, F1=0.279


In [12]:
# ----------------------------------------------------
# نطبق LOF بأفضل بارامترات لقيناها، ونحصل على تصنيف الشذوذ النهائي
best_lof = LocalOutlierFactor(n_neighbors=50, contamination=0.05, n_jobs=-1)
lof_predictions = best_lof.fit_predict(work_df_scaled)

# نضيف النتيجة + عمود Type الأصلي (قبل الترميز، للوضوح) لنفس الصفوف
analysis_df = df[['Type']].copy()  # من الداتا الأصلية df، مو work_df (نبي القيم النصية L/M/H الواضحة)
analysis_df['is_anomaly'] = (lof_predictions == -1)

# ----------------------------------------------------
# نقارن: توزيع Type بين الصفوف الشاذة، مقابل توزيعه الطبيعي بكل الداتا
print("--- توزيع Type بكل الداتا (المرجع) ---")
print(df['Type'].value_counts(normalize=True).round(3))

print("\n--- توزيع Type بين الصفوف المصنّفة 'شاذة' من LOF ---")
print(analysis_df[analysis_df['is_anomaly']]['Type'].value_counts(normalize=True).round(3))

--- توزيع Type بكل الداتا (المرجع) ---
Type
L    0.6
M    0.3
H    0.1
Name: proportion, dtype: float64

--- توزيع Type بين الصفوف المصنّفة 'شاذة' من LOF ---
Type
L    0.524
M    0.318
H    0.158
Name: proportion, dtype: float64


In [14]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ----------------------------------------------------
# الرسمة 1: Torque مقابل Rotational Speed - ملوّنة حسب Machine Failure
colors = ground_truth['Machine failure'].map({0: 'steelblue', 1: 'red'})
axes[0].scatter(df['Rotational speed [rpm]'], df['Torque [Nm]'], c=colors, alpha=0.4, s=10)
axes[0].set_xlabel('Rotational Speed [rpm]')
axes[0].set_ylabel('Torque [Nm]')
axes[0].set_title('Torque vs Speed (Red = Actual Failure)')

# ----------------------------------------------------
# الرسمة 2: Tool wear مقابل Torque - العلاقة المرتبطة بـ Overstrain Failure (ضرب الاثنين)
axes[1].scatter(df['Tool wear [min]'], df['Torque [Nm]'], c=colors, alpha=0.4, s=10)
axes[1].set_xlabel('Tool Wear [min]')
axes[1].set_ylabel('Torque [Nm]')
axes[1].set_title('Tool Wear vs Torque (Red = Actual Failure)')

plt.tight_layout()
plt.savefig('nonlinearity_check.png', dpi=110)
print("تم حفظ الرسمة")

تم حفظ الرسمة


In [15]:
# ----------------------------------------------------
# نضيف عمود Power جديد لكل من الداتا الأصلية والداتا الشغالة
# ملاحظة: في الفيزياء الحقيقية Power = Torque × Angular Velocity (rad/s)
# وبما إنه rotational speed عندنا بالـ rpm، النسبة بينهم ثابتة، فالعلاقة تبقى نفس الشكل التناسبي
work_df['Power'] = work_df['Torque [Nm]'] * work_df['Rotational speed [rpm]']

print("--- إحصائيات عمود Power الجديد ---")
print(work_df['Power'].describe().round(2))

# ----------------------------------------------------
# نعيد الـ Scaling على كل work_df (بما فيها العمود الجديد) من الصفر
scaler_v2 = StandardScaler()
work_df_scaled_v2 = scaler_v2.fit_transform(work_df)
work_df_scaled_v2 = pd.DataFrame(work_df_scaled_v2, columns=work_df.columns, index=work_df.index)

print("\nأعمدة الداتا الجديدة:", work_df.columns.tolist())

--- إحصائيات عمود Power الجديد ---
count    10000.00
mean     59967.15
std      10193.09
min      10966.80
25%      53105.40
50%      59883.90
75%      66873.75
max      99980.40
Name: Power, dtype: float64

أعمدة الداتا الجديدة: ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Power']


In [16]:
# ----------------------------------------------------
# نعيد GridSearch بس على LOF (الأفضل سابقًا) بالداتا الجديدة (مع Power)
best_lof_score_v2, best_lof_params_v2 = 0, None
for n_neighbors in [10, 20, 30, 50]:
    for contamination in [0.02, 0.034, 0.05, 0.1]:
        model = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination, n_jobs=-1)
        pred = model.fit_predict(work_df_scaled_v2)
        score = evaluate_anomaly_detection(pred, actual)
        if score > best_lof_score_v2:
            best_lof_score_v2, best_lof_params_v2 = score, {'n_neighbors': n_neighbors, 'contamination': contamination}

print(f"LOF (مع Power) أفضل: {best_lof_params_v2}, F1={best_lof_score_v2:.3f}")
print(f"LOF (بدون Power) كان: n_neighbors=50, contamination=0.05, F1=0.279")

LOF (مع Power) أفضل: {'n_neighbors': 50, 'contamination': 0.05}, F1=0.277
LOF (بدون Power) كان: n_neighbors=50, contamination=0.05, F1=0.279


In [17]:
# ----------------------------------------------------
# نحسب الوسيط (median) لـ Power من القيم الأصلية (قبل أي Scaling)
power_median = work_df['Power'].median()
print(f"القيمة الوسطية لـ Power: {power_median:.2f}")

# ----------------------------------------------------
# Feature جديد: البُعد المطلق عن الوسيط
# هيك الحالتين المتطرفتين (Power عالي جداً أو منخفض جداً) يصيروا "متشابهين" (بعيدين عن الوسط بنفس الاتجاه)
work_df['Power_deviation'] = np.abs(work_df['Power'] - power_median)

print("\n--- إحصائيات العمود الجديد ---")
print(work_df['Power_deviation'].describe().round(2))

# ----------------------------------------------------
# نعيد Scaling على كل work_df (بما فيها Power_deviation) من الصفر
scaler_v3 = StandardScaler()
work_df_scaled_v3 = scaler_v3.fit_transform(work_df)
work_df_scaled_v3 = pd.DataFrame(work_df_scaled_v3, columns=work_df.columns, index=work_df.index)

# ----------------------------------------------------
# نعيد GridSearch على LOF بالداتا الجديدة
best_lof_score_v3, best_lof_params_v3 = 0, None
for n_neighbors in [10, 20, 30, 50]:
    for contamination in [0.02, 0.034, 0.05, 0.1]:
        model = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination, n_jobs=-1)
        pred = model.fit_predict(work_df_scaled_v3)
        score = evaluate_anomaly_detection(pred, actual)
        if score > best_lof_score_v3:
            best_lof_score_v3, best_lof_params_v3 = score, {'n_neighbors': n_neighbors, 'contamination': contamination}

print(f"\nLOF (مع Power_deviation) أفضل: {best_lof_params_v3}, F1={best_lof_score_v3:.3f}")
print(f"LOF (مع Power الخام) كان: F1=0.277")
print(f"LOF (بدون Power إطلاقاً) كان: F1=0.279")

القيمة الوسطية لـ Power: 59883.90

--- إحصائيات العمود الجديد ---
count    10000.00
mean      8108.69
std       6176.47
min          1.90
25%       3206.85
50%       6885.60
75%      11618.25
max      48917.10
Name: Power_deviation, dtype: float64

LOF (مع Power_deviation) أفضل: {'n_neighbors': 50, 'contamination': 0.1}, F1=0.232
LOF (مع Power الخام) كان: F1=0.277
LOF (بدون Power إطلاقاً) كان: F1=0.279


In [20]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# ----------------------------------------------------
# نستخدم القيم الخام الموحّدة بس (بدون Power أو أي عمود هندسي مضاف) - كما اتفقنا
X_input = work_df_scaled.values  # work_df_scaled الأصلي: Type + 5 قراءات = 6 أعمدة
n_features = X_input.shape[1]

print(f"عدد الأعمدة المدخلة: {n_features}, عدد الصفوف: {X_input.shape[0]}")

# ----------------------------------------------------
# دالة تبني وتدرب Autoencoder بحجم Bottleneck معيّن، وترجع خطأ إعادة البناء لكل صف
def build_and_train_autoencoder(X, bottleneck_size, epochs=50):
    n_feat = X.shape[1]
    
    # المُشفّر (Encoder): يضغط تدريجيًا من n_feat لحد bottleneck_size
    autoencoder = keras.Sequential([
        layers.Input(shape=(n_feat,)),
        layers.Dense(4, activation='relu'),
        layers.Dense(bottleneck_size, activation='relu', name='bottleneck'),
        # المُفكّك (Decoder): يعيد التوسّع تدريجيًا لحد n_feat مرة ثانية
        layers.Dense(4, activation='relu'),
        layers.Dense(n_feat, activation='linear')  # linear لأنه القيم بعد Scaling ممكن تكون سالبة
    ])
    
    autoencoder.compile(optimizer='adam', loss='mse')
    
    # ----------------------------------------------------
    # نلاحظ: X كمدخل وX نفسها كهدف (Target) - الشبكة بتحاول "تعيد إنتاج" مدخلاتها
    autoencoder.fit(X, X, epochs=epochs, batch_size=32, verbose=0, validation_split=0.1)
    
    # ----------------------------------------------------
    # نحسب خطأ إعادة البناء (Reconstruction Error) لكل صف
    X_reconstructed = autoencoder.predict(X, verbose=0)
    reconstruction_error = np.mean((X - X_reconstructed) ** 2, axis=1)
    
    return reconstruction_error

# ----------------------------------------------------
# نجرب 3 أحجام مختلفة للـ Bottleneck: 1, 2, 3
results_autoencoder = {}

for bottleneck_size in [1, 2, 3]:
    print(f"\nتدريب Autoencoder بـ Bottleneck = {bottleneck_size}...")
    errors = build_and_train_autoencoder(X_input, bottleneck_size)
    
    # ----------------------------------------------------
    # نجرب عدة نسب قطع (percentile) لتحديد "شو الخطأ الكبير كفاية ليعتبر شذوذ"
    best_score, best_pct = 0, None
    for pct in [90, 93, 95, 97, 99]:
        threshold = np.percentile(errors, pct)
        pred = np.where(errors > threshold, -1, 1)
        score = evaluate_anomaly_detection(pred, actual)
        if score > best_score:
            best_score, best_pct = score, pct
    
    results_autoencoder[bottleneck_size] = {'F1': best_score, 'percentile': best_pct}
    print(f"Bottleneck={bottleneck_size}: أفضل F1={best_score:.3f} عند percentile={best_pct}")

print("\n--- ملخص كل الأحجام ---")
for size, res in results_autoencoder.items():
    print(f"Bottleneck={size}: F1={res['F1']:.3f}")

print(f"\nللمقارنة - أفضل نتيجة سابقة (LOF بدون Power): F1=0.279")

عدد الأعمدة المدخلة: 6, عدد الصفوف: 10000

تدريب Autoencoder بـ Bottleneck = 1...
Bottleneck=1: أفضل F1=0.152 عند percentile=93

تدريب Autoencoder بـ Bottleneck = 2...
Bottleneck=2: أفضل F1=0.137 عند percentile=90

تدريب Autoencoder بـ Bottleneck = 3...
Bottleneck=3: أفضل F1=0.235 عند percentile=97

--- ملخص كل الأحجام ---
Bottleneck=1: F1=0.152
Bottleneck=2: F1=0.137
Bottleneck=3: F1=0.235

للمقارنة - أفضل نتيجة سابقة (LOF بدون Power): F1=0.279


In [21]:
# ----------------------------------------------------
# نطبق أفضل LOF (n_neighbors=50, contamination=0.05) ونحصل على تصنيف الشذوذ
best_lof_final = LocalOutlierFactor(n_neighbors=50, contamination=0.05, n_jobs=-1)
lof_pred_final = best_lof_final.fit_predict(work_df_scaled)
detected_anomaly = (lof_pred_final == -1).astype(int)

# ----------------------------------------------------
# نحسب F1 لكل نوع عطل من الخمسة لحاله (كل وحدة عمود 0/1 مستقل بـ ground_truth)
failure_types = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

print("--- F1 وعدد الحالات لكل نوع عطل لحاله ---")
for ftype in failure_types:
    actual_type = ground_truth[ftype].values
    n_cases = actual_type.sum()
    
    if n_cases > 0:
        score = f1_score(actual_type, detected_anomaly)
        # كمان نحسب Recall لحاله - "من كل حالات هذا العطل، كم اكتشفنا؟"
        recall = (detected_anomaly[actual_type == 1] == 1).mean()
        print(f"{ftype}: عدد الحالات={n_cases}, F1={score:.3f}, Recall={recall:.3f}")
    else:
        print(f"{ftype}: عدد الحالات=0 (غير موجود بالداتا)")

--- F1 وعدد الحالات لكل نوع عطل لحاله ---
TWF: عدد الحالات=46, F1=0.029, Recall=0.174
HDF: عدد الحالات=115, F1=0.068, Recall=0.183
PWF: عدد الحالات=95, F1=0.235, Recall=0.737
OSF: عدد الحالات=98, F1=0.134, Recall=0.408
RNF: عدد الحالات=19, F1=0.000, Recall=0.000


In [22]:
# ----------------------------------------------------
# نعيد حساب F1 "المُعدّل" - نستثني RNF من التقييم (لأنه عشوائي بالتعريف، غير قابل للاكتشاف من القراءات)
# نبني target جديد: شاذ حقيقي = أي عطل ما عدا RNF لحاله
adjusted_actual = ((ground_truth['TWF'] + ground_truth['HDF'] + 
                     ground_truth['PWF'] + ground_truth['OSF']) > 0).astype(int)

adjusted_f1 = f1_score(adjusted_actual, detected_anomaly)
print(f"F1 الأصلي (يشمل RNF): 0.279")
print(f"F1 المُعدّل (بدون RNF): {adjusted_f1:.3f}")

F1 الأصلي (يشمل RNF): 0.279
F1 المُعدّل (بدون RNF): 0.282


In [23]:
# نعيد GridSearch للأربعة خوارزميات، بس على work_df_scaled_v2 (يشمل Power)
algorithms_results_with_power = {}

# Isolation Forest
best_score, best_params = 0, None
for contamination in [0.02, 0.034, 0.05, 0.1]:
    model = IsolationForest(contamination=contamination, random_state=42, n_jobs=-1)
    pred = model.fit_predict(work_df_scaled_v2)
    score = evaluate_anomaly_detection(pred, actual)
    if score > best_score:
        best_score, best_params = score, {'contamination': contamination}
algorithms_results_with_power['IsolationForest'] = (best_score, best_params)

# DBSCAN
best_score, best_params = 0, None
for eps in [0.8, 1.0, 1.2, 1.5, 2.0]:
    for min_samples in [5, 10, 15, 20]:
        model = DBSCAN(eps=eps, min_samples=min_samples)
        pred = model.fit_predict(work_df_scaled_v2)
        score = evaluate_anomaly_detection(pred, actual)
        if score > best_score:
            best_score, best_params = score, {'eps': eps, 'min_samples': min_samples}
algorithms_results_with_power['DBSCAN'] = (best_score, best_params)

# K-Means
best_score, best_params = 0, None
for k in [2, 3, 4, 5, 6]:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(work_df_scaled_v2)
    distances = np.min(model.transform(work_df_scaled_v2), axis=1)
    for pct in [90, 93, 95, 97, 99]:
        threshold = np.percentile(distances, pct)
        pred = np.where(distances > threshold, -1, 1)
        score = evaluate_anomaly_detection(pred, actual)
        if score > best_score:
            best_score, best_params = score, {'k': k, 'percentile': pct}
algorithms_results_with_power['KMeans'] = (best_score, best_params)

print("--- النتائج مع عمود Power ---")
for name, (score, params) in algorithms_results_with_power.items():
    print(f"{name}: F1={score:.3f}, {params}")

print(f"\n--- للمقارنة، بدون Power ---")
print(f"IsolationForest: F1=0.188 | DBSCAN: F1=0.218 | KMeans: F1=0.241 | LOF: F1=0.279")

--- النتائج مع عمود Power ---
IsolationForest: F1=0.269, {'contamination': 0.05}
DBSCAN: F1=0.259, {'eps': 0.8, 'min_samples': 5}
KMeans: F1=0.304, {'k': 3, 'percentile': 97}

--- للمقارنة، بدون Power ---
IsolationForest: F1=0.188 | DBSCAN: F1=0.218 | KMeans: F1=0.241 | LOF: F1=0.279
